In [ ]:
%load_ext blackcellmagic
# %%black -l 120
%load_ext autoreload
%autoreload 2

In [ ]:
import jax
import time
from slimbbf.algorithms.bbf import BBF
from slimbbf.environments.atari import AtariEnv
from slimbbf.sample_collection.utils import select_action

def one_eval(env, agent, key):
    env.reset_with_noop(jax.random.PRNGKey(0))
    
    return_ = 0
    terminal = False
    epsilon_fn = lambda step: 0.01
    
    while not terminal and env.n_steps < 2 * 2_700:
        key, seletion_key = jax.random.split(key)
        action = jax.block_until_ready(
            select_action(agent.best_action, agent.params, env.state, seletion_key, env.n_actions, epsilon_fn, env.n_steps)
        )
        reward, terminal = env.step(action)
        return_ += reward

    return return_


env = AtariEnv("DoubleDunk", sticky_actions=False)
agent = BBF(        
    jax.random.PRNGKey(0),
    (env.state_height, env.state_width, env.n_stacked_frames),
    env.n_actions,
    n_bins=51,
    features=[64, 128, 128, 2048],
    learning_rate=0.001,
    min_gamma=0.97,
    max_gamma=0.997,
    min_update_horizon=3,
    max_update_horizon=10,
    gamma_horizon_decay_steps=2000,
    tau=0.005,
    spr_steps=5,
)

t_start = time.time()
return_ = one_eval(env, agent, jax.random.PRNGKey(0))
print(f"{time.time() - t_start} sec for a return of {return_}")

In [ ]:
import jax
import time
import numpy as np
from slimbbf.algorithms.bbf import BBF
from slimbbf.environments.atari_eval import AtariEval
from slimbbf.sample_collection.utils import select_action_eval


def one_eval(env, agent, key):
    env.reset_with_noop(jax.random.PRNGKey(0))
    episode_termination = env.termination_mask  # needed for considering rewards,length until env.termination_mask
    episode_returns = np.zeros(env.n_envs)
    episode_lengths = np.zeros(env.n_envs)
    epsilon_fn = lambda _: 0.001
    
    while not episode_termination.all() and env.n_steps < 2 * 2_700:
        key, seletion_key = jax.random.split(key)
        
        actions = jax.block_until_ready(
            select_action_eval(agent.best_action, agent.params, env.states, seletion_key, env.n_actions, epsilon_fn)
        )
        rewards = env.step(np.array(actions))  # episode.termination changes here, so we use episode_termination
        
        episode_returns += rewards * (1 - episode_termination)
        episode_lengths += 1 - episode_termination
        episode_termination = env.termination_mask

    return episode_returns.tolist(), episode_lengths.tolist()

env = AtariEval("DoubleDunk", sticky_actions=False, n_envs=100)
agent = BBF(        
    jax.random.PRNGKey(0),
    (env.state_height, env.state_width, env.n_stacked_frames),
    env.n_actions,
    n_bins=51,
    features=[64, 128, 128, 2048],
    learning_rate=0.001,
    min_gamma=0.97,
    max_gamma=0.997,
    min_update_horizon=3,
    max_update_horizon=10,
    gamma_horizon_decay_steps=2000,
    tau=0.005,
    spr_steps=5,
)

t_start = time.time()
returns,  lenghts = one_eval(env, agent, jax.random.PRNGKey(0))
print(f"{time.time() - t_start} sec for a return of {np.mean(returns)} and {np.mean(lenghts)} steps")